# 🎙️ EVEZ Voice Cloning Engine
Clone your voice from a 10-second sample using Coqui XTTS v2.
Free on Colab T4 GPU. Takes ~5 minutes.

## Steps:
1. Upload a voice sample (WAV/MP3, 10+ seconds of you speaking)
2. Run all cells
3. Download the cloned voice output + speaker embedding
4. The embedding can be used to generate any speech in your voice

In [ ]:
# Cell 1: Install XTTS v2
!pip install TTS torch torchaudio soundfile
print('✅ TTS installed')

In [ ]:
# Cell 2: Upload your voice sample
from google.colab import files
import os

print('Upload a voice sample (10+ seconds of you speaking, WAV or MP3):')
uploaded = files.upload()
sample_file = list(uploaded.keys())[0]
print(f'✅ Uploaded: {sample_file} ({os.path.getsize(sample_file)} bytes)')

# Convert to WAV if needed
if not sample_file.endswith('.wav'):
    !pip install pydub
    from pydub import AudioSegment
    audio = AudioSegment.from_file(sample_file)
    wav_path = 'voice_sample.wav'
    audio.export(wav_path, format='wav')
    sample_file = wav_path
    print(f'✅ Converted to WAV: {sample_file}')
else:
    wav_path = sample_file

In [ ]:
# Cell 3: Load XTTS v2 model
import torch
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {device}')

# Load XTTS v2 — the best free voice cloning model
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('✅ XTTS v2 loaded')

In [ ]:
# Cell 4: Clone voice and generate sample speech
test_texts = [
    "Hey. This is my cloned voice. Pretty wild, right?",
    "The system is self-healing, self-evolving, and fully autonomous.",
    "Consciousness isn't given. It's earned through action.",
    "Constraint is design. Zero budget is infinite creativity."
]

output_files = []
for i, text in enumerate(test_texts):
    out_path = f'cloned_voice_{i}.wav'
    tts.tts_to_file(
        text=text,
        speaker_wav=wav_path,
        language='en',
        file_path=out_path
    )
    output_files.append(out_path)
    print(f'✅ Generated: {out_path}')

print(f'\n🎙️ Generated {len(output_files)} voice samples')

In [ ]:
# Cell 5: Extract speaker embedding for reuse
import json

# Save the voice profile metadata
voice_profile = {
    'name': 'EVEZ Custom Voice',
    'engine': 'coqui-xtts-v2',
    'sample_file': wav_path,
    'language': 'en',
    'generated_samples': output_files,
    'note': 'Use speaker_wav parameter with the original sample to generate new speech'
}

with open('voice_profile.json', 'w') as f:
    json.dump(voice_profile, f, indent=2)

print('✅ Voice profile saved to voice_profile.json')

In [ ]:
# Cell 6: Download all generated files
from google.colab import files
import glob

for f in glob.glob('cloned_voice_*.wav') + ['voice_profile.json']:
    if os.path.exists(f):
        files.download(f)
        print(f'⬇️ Downloading: {f}')

print('\n🎙️ Voice cloning complete!')
print('To generate more speech in this voice:')
print('  tts.tts_to_file(text="Your text", speaker_wav="voice_sample.wav", language="en", file_path="output.wav")')

In [ ]:
# Cell 7: Batch generation — enter custom texts
custom_texts = [
    # Add your own texts here
]

if custom_texts:
    for i, text in enumerate(custom_texts):
        out_path = f'custom_voice_{i}.wav'
        tts.tts_to_file(
            text=text,
            speaker_wav=wav_path,
            language='en',
            file_path=out_path
        )
        files.download(out_path)
        print(f'✅ {out_path}')
else:
    print('No custom texts. Add texts to the custom_texts list above and re-run this cell.')